In [ ]:
import socket
import pyvisa
import time

print("="*70)
print("BNC 765 CONNECTION DIAGNOSTIC - COMPREHENSIVE")
print("="*70)

# Get IP
BNC_IP = input("\nEnter BNC 765 IP address: ").strip()
if not BNC_IP:
    # BNC_IP = '169.254.209.156' # Kepler pulser
    BNC_IP = '169.254.125.69' #HMMB pulser

print(f"\nTarget IP: {BNC_IP}\n")

# STEP 1: Ping test
print("-"*70)
print("STEP 1: Basic Network Test")
print("-"*70)

import subprocess
import platform

param = '-n' if platform.system().lower() == 'windows' else '-c'
try:
    result = subprocess.run(['ping', param, '1', BNC_IP],
                          capture_output=True, timeout=3)
    if result.returncode == 0:
        print(f"✓ Device responds to ping")
    else:
        print(f"✗ Device does not respond to ping")
        print(f"  Check: IP address, network cable, device power")
except:
    print("⚠ Could not perform ping test")

# STEP 2: Port scan
print("\n" + "-"*70)
print("STEP 2: Port Scan")
print("-"*70 + "\n")

ports = [5025, 5024, 5000, 5001, 111, 23, 80, 9221]
open_ports = []

for port in ports:
    print(f"Port {port:5d}...", end=" ")
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(1)
        if sock.connect_ex((BNC_IP, port)) == 0:
            print("✓ OPEN")
            open_ports.append(port)
        else:
            print("✗ closed")
        sock.close()
    except:
        print("✗ error")

if not open_ports:
    print("\n✗ No open ports found!")
    print("Device may be:")
    print("  - Powered off")
    print("  - Wrong IP address")
    print("  - Different network/subnet")
    print("  - Firewall blocking")
    exit()

print(f"\n✓ Found {len(open_ports)} open port(s): {open_ports}")

# STEP 3: Test raw socket on each port
print("\n" + "-"*70)
print("STEP 3: Raw Socket Test")
print("-"*70)

working_port = None
for port in open_ports:
    print(f"\nTesting port {port}...")
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(3)
        sock.connect((BNC_IP, port))

        # Try SCPI command
        sock.send(b'*IDN?\n')
        time.sleep(0.1)
        response = sock.recv(1024)

        if response:
            print(f"  ✓ Got response: {response.decode().strip()}")
            working_port = port
            sock.close()
            break
        else:
            print(f"  ✗ No response")

        sock.close()
    except Exception as e:
        print(f"  ✗ Error: {e}")

if working_port:
    print(f"\n✓✓✓ DEVICE RESPONDS ON PORT {working_port}! ✓✓✓")

    # STEP 4: Test VISA
    print("\n" + "-"*70)
    print("STEP 4: VISA Connection Test")
    print("-"*70 + "\n")

    rm = pyvisa.ResourceManager()

    addresses = [
        f'TCPIP::{BNC_IP}::{working_port}::SOCKET',
        f'TCPIP::{BNC_IP}::INST',
        f'TCPIP0::{BNC_IP}::{working_port}::SOCKET',
    ]

    for addr in addresses:
        print(f"Trying: {addr}")
        try:
            device = rm.open_resource(addr)
            device.timeout = 5000
            device.write_termination = '\n'
            device.read_termination = '\n'

            idn = device.query('*IDN?')
            print(f"  ✓ SUCCESS!")
            print(f"  Device: {idn.strip()}")

            print(f"\n{'='*70}")
            print(f"✓✓✓ WORKING CONNECTION FOUND! ✓✓✓")
            print(f"{'='*70}")
            print(f"\nUse this address in your code:")
            print(f"  pulse_gen = rm.open_resource('{addr}')")
            print(f"  pulse_gen.timeout = 5000")
            print(f"  pulse_gen.write_termination = '\\n'")
            print(f"  pulse_gen.read_termination = '\\n'")

            device.close()
            break

        except Exception as e:
            print(f"  ✗ Failed: {e}\n")
else:
    print("\n✗ Could not communicate with device on any port")
    print("\nNext steps:")
    print("  1. Check BNC 765 manual for network configuration")
    print("  2. Look for 'LXI' or 'SCPI Socket' settings on device")
    print("  3. Try web interface at http://" + BNC_IP)
    print("  4. Contact BNC support for network setup info")

BNC 765 CONNECTION DIAGNOSTIC - COMPREHENSIVE


In [ ]:
import pyvisa

print("="*70)
print("CONNECTING TO BNC 765 VIA VXI-11")
print("="*70 + "\n")
BNC_IP = '169.254.209.156'

# Initialize VISA
rm = pyvisa.ResourceManager()
print(f"VISA Backend: {rm}\n")

# VXI-11 address formats to try
addresses = [
    f'TCPIP::{BNC_IP}::INSTR',           # Standard VXI-11
    f'TCPIP0::{BNC_IP}::INSTR',          # With board number
    f'TCPIP::{BNC_IP}::inst0::INSTR',    # Explicit instrument
    f'TCPIP::{BNC_IP}::gpib0,1::INSTR',  # GPIB-style
]

pulse_gen = None

for address in addresses:
    print(f"Trying: {address}")

    try:
        device = rm.open_resource(address)
        device.timeout = 10000  # VXI-11 can be slow, use longer timeout

        print(f"  Connection opened, querying device...")

        # Try *IDN?
        idn = device.query('*IDN?')

        print(f"\n✓✓✓ SUCCESS! ✓✓✓")
        print(f"  Address: {address}")
        print(f"  Device: {idn.strip()}")

        pulse_gen = device
        break

    except pyvisa.errors.VisaIOError as e:
        print(f"  ✗ VISA error: {e}\n")
    except Exception as e:
        print(f"  ✗ Error: {type(e).__name__}: {e}\n")

if pulse_gen:
    print("\n" + "="*70)
    print("TESTING BASIC COMMANDS")
    print("="*70 + "\n")

    # Test some commands
    commands = [
        ('*IDN?', 'Device ID'),
        ('*RST', 'Reset (write only)'),
        (':PULSE1:WIDTH?', 'CH1 Pulse Width'),
        (':PULSE1:STATE?', 'CH1 State'),
    ]

    for cmd, desc in commands:
        print(f"{desc}:")
        print(f"  Command: {cmd}")
        try:
            if '?' in cmd:
                result = pulse_gen.query(cmd)
                print(f"  Response: {result.strip()}")
            else:
                pulse_gen.write(cmd)
                print(f"  ✓ Sent")
        except Exception as e:
            print(f"  ✗ Error: {e}")
        print()

    print("="*70)
    print("✓ BNC 765 is ready to use!")
    print("="*70)

    # Keep connection open for testing
    input("\nPress Enter to close connection...")
    pulse_gen.close()

else:
    print("\n" + "="*70)
    print("✗ Could not connect via VXI-11")
    print("="*70)
    print("\nTroubleshooting VXI-11:")
    print("  1. VXI-11 may need specific VISA backend")
    print("  2. Try NI-VISA or Keysight IO Libraries")
    print("  3. Check BNC 765 network settings for VXI-11 configuration")

rm.close()

In [ ]:
import pyvisa

print("="*70)
print("DISCOVERING BNC 765 COMMANDS")
print("="*70 + "\n")

# Connect
rm = pyvisa.ResourceManager()
pulse_gen = rm.open_resource('TCPIP::169.254.209.156::INSTR')
pulse_gen.timeout = 10000

print(f"Connected: {pulse_gen.query('*IDN?').strip()}\n")

# Test different command variations for Channel 1
print("Testing command variations for CH1:\n")

command_variations = [
    # Width commands
    (':PULSE1:WIDTH?', 'Pulse Width'),
    (':PULS1:WIDT?', 'Width (short)'),
    ('PULSE1:WIDTH?', 'Width (no colon)'),

    # Period/Frequency commands
    (':PULSE1:PERIOD?', 'Period'),
    (':PULSE1:PER?', 'Period (short)'),
    (':PULS1:PER?', 'Period (short v2)'),
    (':PULSE1:FREQ?', 'Frequency'),
    (':PULSE1:FREQUENCY?', 'Frequency (full)'),
    ('PULSE1:PER?', 'Period (no colon)'),

    # Amplitude commands
    (':PULSE1:AMPLITUDE?', 'Amplitude'),
    (':PULSE1:AMP?', 'Amplitude (short)'),
    (':PULS1:AMPL?', 'Amplitude (short v2)'),

    # Delay commands
    (':PULSE1:DELAY?', 'Delay'),
    (':PULSE1:DEL?', 'Delay (short)'),
    (':PULS1:DEL?', 'Delay (short v2)'),

    # Polarity commands
    (':PULSE1:POLARITY?', 'Polarity'),
    (':PULSE1:POL?', 'Polarity (short)'),
    (':PULS1:POL?', 'Polarity (short v2)'),

    # State commands
    (':PULSE1:STATE?', 'State'),
    (':PULSE1:STAT?', 'State (short)'),
    (':OUTPUT1?', 'Output state'),
    (':OUTP1?', 'Output (short)'),
    ('OUTPUT1:STATE?', 'Output state (alt)'),

    # Other possibilities
    (':PULSE1:RATE?', 'Rate'),
    (':PULSE1:TIME?', 'Time'),
]

working_commands = {}

for cmd, description in command_variations:
    print(f"{description:25s} ({cmd:30s})...", end=" ")

    try:
        result = pulse_gen.query(cmd).strip()
        print(f"✓ {result}")
        working_commands[description] = (cmd, result)
    except pyvisa.errors.VisaIOError as e:
        if 'TMO' in str(e):
            print("✗ Timeout")
        elif 'INV_EXPR' in str(e):
            print("✗ Invalid command")
        else:
            print(f"✗ {e}")
    except Exception as e:
        print(f"✗ {type(e).__name__}")

# Summary
print("\n" + "="*70)
print("WORKING COMMANDS SUMMARY")
print("="*70 + "\n")

for desc, (cmd, result) in working_commands.items():
    print(f"{desc:25s}: {cmd:30s} → {result}")

# Test write commands
print("\n" + "="*70)
print("TESTING WRITE COMMANDS")
print("="*70 + "\n")

write_tests = [
    (':PULSE1:WIDTH 200E-9', 'Set width to 200ns'),
    (':PULSE1:PERIOD 2E-6', 'Set period to 2μs'),
    (':PULSE1:FREQ 500E3', 'Set frequency to 500kHz'),
]

print("Testing which format works for setting period/frequency:\n")

for cmd, description in write_tests:
    print(f"{description}:")
    print(f"  Command: {cmd}")

    try:
        pulse_gen.write(cmd)
        print(f"  ✓ Write accepted")

        # Try to verify
        if 'PERIOD' in cmd:
            try:
                result = pulse_gen.query(':PULSE1:PERIOD?')
                print(f"  Verified: {result.strip()}")
            except:
                print(f"  Could not verify")
        elif 'FREQ' in cmd:
            try:
                result = pulse_gen.query(':PULSE1:FREQ?')
                print(f"  Verified: {result.strip()}")
            except:
                print(f"  Could not verify")

    except Exception as e:
        print(f"  ✗ Error: {e}")

    print()

pulse_gen.close()
print("="*70)